# TEACH FOREACH

*Converted from the Poplog teaching corpus (`pop/teach/foreach`) by `tools/jupyter/teach2nb.py`. Every code cell runs in one persistent, natively-compiled Pop-11 session — edit and re-run them freely; a mishap is survivable and often intentional.*

```
                                     Updated to use "!" October 1996
```

This file introduces the following looping syntax for doing something
with all items in the database that match a given pattern:

```
foreach <pattern> do <actions> endforeach
```

or

```
foreach ! <pattern> do <actions> endforeach
```

## Prerequisites for this file

This file assumes that you are familiar with the use of MATCHES to
compare a list and a pattern (e.g. see TEACH * MATCHES), and that you
already know that the Pop-11 database is a list of lists that can be
searched using procedures based on the matcher (as in TEACH * DATABASE).

You should be familiar with the role of the variable prefixes "?" and
"??" in patterns, to SET the value of a variable, and the role of the
single and double uparrows "^" and "^^" to USE the existing value of a
variable. (See TEACH * ARROW.)

As explained in TEACH * MATCHES, if you define a procedure that uses a
pattern containing pattern variables preceded by "?" or "??", then you
should use "!" in front of the pattern, so that the pattern variables
can be used as lvars, reducing the risk of unwanted interactions between
procedures using the same variables.

## What is foreach for?

Suppose you have a collection of facts stored in the database about
people, their relations, their jobs, their possessions, etc. You can use
a pattern to select items from the database. E.g. [?person isa teacher]
would match database items such as

```
[suzie isa teacher]
[john isa teacher]
```

If you wish to find only one item matching the pattern then you can use
present or lookup (explained in TEACH * DATABASE). If you wish to find
all the matching items, e.g. all the people who are teachers, then you
can use foreach, e.g. using a form like this, which will perform some
action on each person recorded as a teacher.

In [1]:
foreach [?person isa teacher] do ....person.... endforeach;

;;; DECLARING VARIABLE person
;;; FILE     :  /Users/dkords/.cache/pop11-skill/session-jupyter79251/spool2.p
;;; 	  LINE NUMBER:  1
;;; PRINT DOING
;;; DOING    :  . foreach Do_expr_syntax_opener Comp_expr_seq skill_run
;;; 	pop_setpop_compiler


🌈 pop 6.43 ms │ session 18.48 ms over 2 runs │ all-time 27781.30 s


or, inside a procedure definition, using the "!" prefix

In [2]:
foreach ! [?person isa teacher] do ....person.... endforeach;

🌈 pop 6.00 ms │ session 24.48 ms over 3 runs │ all-time 27781.31 s


Note the occurrence of "?" before person means that person is a pattern
variable. It will therefore need to be declared as a variable. Outside a
procedure definition you can use

In [3]:
vars person;

🌈 pop 6.58 ms │ session 31.06 ms over 4 runs │ all-time 27781.32 s


Inside a procedure definition use "lvars".

## An example database

Lets demonstrate with a database of information about some people. You
may find it convenient to put the following examples into a file of your
own called something like database.p, or foreach.p. You can delete the
file later on. Insert this procedure definition:

In [4]:
define people();
    ;;; clear the database and set up a collection of "initial" facts
    []  -> database;
    add([joe isa man]);
    add([jill isa woman]);
    add([joe lives_in london]);
    add([jill lives_in brighton]);
    add([bill isa man]);
    add([sue isa woman]);
    add([bill lives_in london]);
    add([sue lives_in paris]);
enddefine;

🌈 pop 6.62 ms │ session 37.69 ms over 5 runs │ all-time 27781.32 s


You could extend this procedure with more examples of the same general
kind, if you wish.

Note 1. We could use "is a" as two separate words, and "lives in" as two
separate words. That would certainly work, but would require more
storage space, and would slightly slow down searching the database.

Note 2. Instead of all those separate add(<list>) commands you can use a
single command calling the procedure alladd with a list of lists, e.g.

```
alladd([
        [joe isa man]
        [jill isa woman]
        ....
        [sue lives_in paris]])
```

We'll use this procedure every time we want to start off with a new database.
Try it out:

In [5]:
;;; Empty the database
[] -> database;
database ==>
;;; Initialise the database.
people();
database ==>

** []
** [[sue lives_in paris]
    [bill lives_in london]
    [sue isa woman]
    [bill isa man]
    [jill lives_in brighton]
    [joe lives_in london]
    [jill isa woman]
    [joe isa man]]


🌈 pop 6.48 ms │ session 44.17 ms over 6 runs │ all-time 27781.33 s


## The procedure present finds one thing only

We have information about several women in the database, but if you do

In [6]:
vars x;
present([??x isa woman])=>
x =>

** <true>
** [sue]


🌈 pop 5.99 ms │ session 50.16 ms over 7 runs │ all-time 27781.34 s


and then do it again

In [7]:
present([??x isa woman])=>
x =>

** <true>
** [sue]


🌈 pop 6.50 ms │ session 56.65 ms over 8 runs │ all-time 27781.34 s


you'll see that it always finds the same thing (provided the database has not
changed in between).

That's fine if you just want to get the name of any one woman. But suppose
you want to find the names of all of them?

## foreach iterates selectively over the whole database

The POP11 syntax word 'FOREACH' can be used to solve the problem.
Try the following:

In [8]:
vars x;
foreach [??x isa woman] in database do x => endforeach;

** [sue]
** [jill]


🌈 pop 6.45 ms │ session 63.10 ms over 9 runs │ all-time 27781.35 s


Every time the pattern matches a database item, the instruction "x=>" in
the body of the loop will be obeyed. (This is called a loop because it
does the same instruction repeatedly, though with a different value for
"x" each time.)

Inside a procedure definition you would do this

In [9]:
lvars x;
foreach ! [??x isa woman] in database do x => endforeach;

** [sue]
** [jill]


🌈 pop 5.66 ms │ session 68.76 ms over 10 runs │ all-time 27781.35 s


## The syntax of foreach

FOREACH is used in the format:

```
FOREACH <pattern> IN <list of lists> DO <action> ENDFOREACH
```

It uses MATCHES to compare each of the lists against the <pattern>, and every
time the match is successful it does the <action>.

When the <list of lists> is the DATABASE, you can leave out the 'IN....' bit.

E.g. try

In [10]:
foreach [??x isa woman] do x => endforeach;

** [sue]
** [jill]


🌈 pop 6.43 ms │ session 75.19 ms over 11 runs │ all-time 27781.36 s


I.e. since you don't say in WHAT list of lists, it assumes you mean in the
database.

Try getting foreach to print out all the names of all the MEN.

## Where do the men live?

Here is how you can make FOREACH print out the home towns of all the men

In [11]:
;;; set up the database with information about people
people();
;;; find where all the men live
vars place, x;
foreach [??x isa man] do
  lookup([^^x lives_in ??place]);
  [the home of ^x is ^^place] =>
endforeach;

** [the home of [bill] is london]
** [the home of [joe] is london]


🌈 pop 5.89 ms │ session 81.08 ms over 12 runs │ all-time 27781.37 s


Try that. Then try doing the same for all the women.

Then try printing out all the people who live in london.

See what difference it makes if you replace "^x" with "^^x" in the last
action.

Also see what difference it makes if you use a single query in the
pattern:

In [12]:
vars place, x;
foreach [?x isa man] do
  lookup([^x lives_in ??place]);
  [the home of ^x is ^^place] =>
endforeach;

** [the home of bill is london]
** [the home of joe is london]


🌈 pop 6.44 ms │ session 87.53 ms over 13 runs │ all-time 27781.37 s


Why did changing the "??x" to "?x" mean that "^^x" had to be changed to
"^x" ?d (If you can't answer that, ask for help. Understanding it is
very important in using the pattern matcher and the database).

You can mark and load the above instruction and it should work, after
you have run the people procedure. However, if the above instruction
were used inside a procedure definition, you could use "!" in front of
all the patterns, without the "vars" declaration. Look closely at the
above and decide which list expressions are patterns that need to be
preceded by "!".

Then compare your decision with the occurrences in this procedure:

In [13]:
define where_men();
    ;;; find where all the men live
    ;;; declare the pattern variables
    lvars x, place;
    foreach ! [?x isa man] do
        lookup( ! [^x lives_in ??place]);
        [the home of ^x is ^^place] =>
    endforeach;
enddefine;

🌈 pop 5.93 ms │ session 93.45 ms over 14 runs │ all-time 27781.38 s


Compile that and then test it

In [14]:
where_men();

** [the home of bill is london]
** [the home of joe is london]


🌈 pop 6.42 ms │ session 99.88 ms over 15 runs │ all-time 27781.39 s


Can you generalise that to define a procedure that takes either the word
"man" or the word "woman" and prints out where all the instances live.

In [15]:
define where_live(type);
    ;;; Find everything that isa type and print out where they live
    [The home of each ^type] =>
    ;;; declare pattern variables
    lvars x, place;
    foreach ! [?x isa ... ] do
        lookup( ! [^x lives_in ??place]);
        [the home of ^x is ^^place] =>
    endforeach;

enddefine;

🌈 pop 6.42 ms │ session 106.30 ms over 16 runs │ all-time 27781.39 s


What should replace the "..." in the foreach line? Why?

Test this after making sure the people database is set up:

In [16]:
people();
where_live("man");
where_live("woman");

** [The home of each man]
** [The home of each woman]


🌈 pop 5.53 ms │ session 111.83 ms over 17 runs │ all-time 27781.40 s


## Making a list of information from database

So far all our FOREACH loops have merely printed something out each time.
Suppose we wanted a procedure which could make a list of all the women, or a
list of all the men. We'd need to go through the database as before searching
for things matching a certain pattern. But each time we found the appropriate
item, instead of printing it out, we can add it to a list.

We'll call our procedure ALLOF. It will take a list like [man] or [woman],
i.e. a list containing what can follow 'isa' in a database entry. And it
will use that to find corresponding individuals. Foreach such individual, it
will add it to a list, which is finally to be returned as a result. Thus:

In [17]:
define allof(type) -> out;
    ;;; type is a list , e.g. [man] or [woman].
    ;;; out, the output variable, will be given a list of words.
    ;;; Start out as the empty list, then build it up
    [] -> out;

    ;;; pattern variable
    lvars person;
    foreach ! [?person isa ^^type] do
        [^person ^^out] -> out
    endforeach;
enddefine;

🌈 pop 6.43 ms │ session 118.27 ms over 18 runs │ all-time 27781.40 s


Type that into your test file and try it out:

In [18]:
allof([man]) =>
allof([woman]) =>

** [joe bill]
** [jill sue]


🌈 pop 6.54 ms │ session 124.81 ms over 19 runs │ all-time 27781.41 s


It's a good idea to put test commands inside comment brackets, thus:

/*

In [19]:
allof([man]) =>
allof([woman]) =>

** [joe bill]
** [jill sue]


🌈 pop 6.50 ms │ session 131.31 ms over 20 runs │ all-time 27781.42 s


*/

## How allof works

To understand this procedure you need to remember how to build lists (e.g. as
in TEACH ARROW).

The line

In [20]:
    [] -> out;

;;; DECLARING VARIABLE out
;;; FILE     :  /Users/dkords/.cache/pop11-skill/session-jupyter79251/spool21.
;;; 	p   LINE NUMBER:  2
;;; PRINT DOING
;;; DOING    :  -> Comp_expr_seq skill_run pop_setpop_compiler


🌈 pop 5.50 ms │ session 136.81 ms over 21 runs │ all-time 27781.42 s


initialises the variable OUT to contain an empty list. Then each time an item
is found in the database matching the pattern

```
    ! [?person isa ^^type]
```

we add the value of PERSON to all the things already in OUT

```
    [^person ^^out]
```

and then assign the new list to be the new value of OUT

```
        -> out
```

When ENDDEFINE is reached, i.e. the procedure finishes running, then the
value of OUT is left on the stack as the result of the procedure. By
then the value of out should be a list of words. In some databases it
could be an empty list, if nothing was found. In that case the body of
the loop would never run. Check that, as follows:

/*

In [21]:
allof([cat]) =>
allof([dog]) =>

** []
** []


🌈 pop 6.53 ms │ session 143.34 ms over 22 runs │ all-time 27781.43 s


*/

Look back at the procedure and make sure you can see how this
description fits it.

## Exercise define allof2

How would you have to change that procedure to make it take the word
"man" or the word "woman" instead of the lists?

Try changing the definition so that it defines a procedure called
allof2, which takes a WORD rather than a LIST as input. What should
replace the "...." below, and why?

In [22]:
define allof2(type) -> out;
    ;;; type is a word, e.g. "man" or "woman".
    ;;; out, the output variable, will be given a list of words.

    lvars person;

    [] -> out;
    foreach ! [?person isa ...] do
        [^person ^^out] -> out
    endforeach;
enddefine;

🌈 pop 5.42 ms │ session 148.76 ms over 23 runs │ all-time 27781.43 s


Test it
/*

In [23]:
people();   ;;; defined above
allof2("man") =>
allof2("woman") =>
allof2("mouse") =>

** []
** []
** []


🌈 pop 6.50 ms │ session 155.25 ms over 24 runs │ all-time 27781.44 s


*/

## An alternative formulation, using [% ... %]

Pop-11 allows the percent symbol to be used in list brackets to collect
together into a list all the things left on the "user stack" in an
instruction.

E.g. try compiling this, or something similar (getting all the commas
and spaces right):

In [24]:
[% 3, 4, 5 + 6 %] =>

[% repeat 6 times [cat on mat] endrepeat %] =>

** [3 4 11]
** [[cat on mat] [cat on mat] [cat on mat] [cat on mat] [cat on mat] [cat
	on mat]]


🌈 pop 6.50 ms │ session 161.75 ms over 25 runs │ all-time 27781.45 s


In each case, the instructions between the percent symbols will be obeyed,
and things will be left on the stack, but [ .... ] will collect them into
a list.

We can use this to redefine ALLOF so that it doesn't have to explicitly
build a bit of the list each time round. Instead, we put the foreach
expression between "%....%" inside list brackets, and let the body of
foreach put what's been found on the stack, each time. Call the new
version ALLOF3:

In [25]:
define allof3(type) -> out;
    ;;; Type is a word. Out is a list of words, i.e. names of persons

    lvars person;

    [%
        foreach ! [?person isa ^type] do
            person
        endforeach
    %] -> out
enddefine;

🌈 pop 6.32 ms │ session 168.07 ms over 26 runs │ all-time 27781.45 s


I.e. the "foreach ... endforeach" loop between % ... % contains
instructions which will put different values of PERSON on the stack, and
at the end the square brackets will make a list, which is then assigned
to OUT.

Compare this version with the previous one. Which you use is a matter of
taste. The only difference is that in the final list the items will be in a
different order from the previous version. Test this.

/*

In [26]:
people();   ;;; defined above
allof3("man") =>
allof3("woman") =>
allof3("mouse") =>

** [bill joe]
** [sue jill]
** []


🌈 pop 5.58 ms │ session 173.65 ms over 27 runs │ all-time 27781.46 s


*/

## Exercise: Finding the occupants of a town

Try to use the ideas just illustrated to define a procedure called
"occupants" which takes the name of a town, e.g. a word like "london" or
"brighton", and then makes a list of names of all the people who
'lives_in' the town.

```
define occupants(place) -> list;
    ;;; place is a word naming a place. Return a list of words
    ;;; naming people who live at the place.
    ........
enddefine;
```

Use a pattern something like [?person lives_in ^place] in the FOREACH loop.

Test your procedure:
/*

In [27]:
occupants("london")=>
occupants("brighton")=>
occupants("berlin")=>

;;; DECLARING VARIABLE occupants
;;; FILE     :  /Users/dkords/.cache/pop11-skill/session-jupyter79251/spool28.
;;; 	p   LINE NUMBER:  2
;;; PRINT DOING
;;; DOING    :  => Comp_expr_seq skill_run pop_setpop_compiler
;;; MISHAP - enp: EXECUTING NON-PROCEDURE
;;; INVOLVING:  <undef occupants>
;;; FILE     :  /Users/dkords/.cache/pop11-skill/session-jupyter79251/spool28.
;;; 	p   LINE NUMBER:  2
;;; PRINT DOING
;;; DOING    :  skill_run pop_setpop_compiler


mishap: ;;; DOING    :  skill_run pop_setpop_compiler

*/

--------------------------------------------------------------------

If you forget the format for FOREACH and you want a reminder later, you can
look at HELP * FOREACH, which gives a summary.

Further examples of the use of FOREACH can be found in TEACH * INFECT

A related facility is explained in HELP * WHICH


HELP * FOREVERY describes forevery, a generalisation of foreach, as it
takes a list of patterns, not just one pattern, and finds all possible
ways of consistently matching items from the list with something in the
database.

--- $poplocal/local/teach/foreach
--- Copyright University of Birmingham 1996. All rights reserved. ------